# open-rknpu walkthrough: compile, inspect, verify, calibrate

An end-to-end, **host-only** tour of the `open-rknpu` compiler for the Rockchip
RV1103/RV1106 NPU (Luckfox Pico Mini B). Every code cell below runs on a laptop or
in Colab: it compiles small ONNX graphs, decodes the containers, runs the profile's
Python integer reference and compares the resulting bytes, and calibrates a graph
with the three selectors. No board, no vendor toolchain and no PyTorch is needed.

The board commands live in the last markdown cell and are **not executed here** -
they need the physical board.

Cells reuse state from the cells above them, exactly like a notebook run: run them
in order (Cell -> Run All).

## 0. Install

In Colab, the badge starts from the repository, so `pip install -e .` installs the
checkout you are reading. Locally, run these two lines in a shell first:

```sh
git clone https://github.com/dnhkng/open-rknpu && cd open-rknpu
python -m pip install -e .          # requires Python >= 3.10, NumPy and ONNX
```

The rest of the notebook only imports `numpy`, `onnx` and `open_rknpu`.

In [ ]:
import hashlib
import json
import tempfile
from dataclasses import fields as dataclass_fields
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, helper as h, numpy_helper as nh

from open_rknpu.calibration import measure
from open_rknpu.chain import native_reference
from open_rknpu.native import native_input_reference
from open_rknpu.quantization import Quantization, reference
from open_rknpu.scheduler import compile_sequence
from open_rknpu.sequence import decode_sequence
from open_rknpu.walk import (chain_walk_reference, join_walk_reference, load_quantizations,
                             parse_chain, parse_join_walk)

FLOAT = TensorProto.FLOAT
PROFILE_KEYS = ("sequence_profile", "profile", "lut_profile", "transposed_profile",
                "depthwise_profile", "elementwise_profile")


def tensor_info(name, shape):
    """A ValueInfoProto for one external tensor."""
    return h.make_tensor_value_info(name, FLOAT, list(shape))


def initializer(name, array):
    """A float32 immutable initializer."""
    return nh.from_array(np.asarray(array), name)


def conv_node(source, output, weights, bias, kernel, pads=None, strides=(1, 1), dilations=(1, 1)):
    """A Conv node with explicit kernel/stride/dilation/padding attributes."""
    if pads is None:
        pads = [kernel // 2] * 4
    return h.make_node("Conv", [source, weights, bias], [output], kernel_shape=[kernel, kernel],
                       pads=list(pads), strides=list(strides), dilations=list(dilations))


def pool_node(kind, source, output):
    """The only verified pool shape: 2x2, stride 2, no padding."""
    return h.make_node(kind, [source], [output], kernel_shape=[2, 2], strides=[2, 2],
                       pads=[0, 0, 0, 0])


def model_graph(nodes, name, inputs, outputs, constants):
    """A checked ModelProto with one opset import and IR version 8."""
    graph = h.make_graph(list(nodes), name, list(inputs), list(outputs), list(constants))
    model = h.make_model(graph, opset_imports=[h.make_opsetid("", 13)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


def quantization_of(entry):
    """Rebuild the emitter's live Quantization from its metadata entry."""
    values = dict(entry.get("quantization", entry))
    names = {field.name for field in dataclass_fields(Quantization)}
    values = {key: value for key, value in values.items() if key in names}
    for key in ("weights", "weight_zero_points", "weight_scales", "biases", "channel_multipliers"):
        if key in values:
            values[key] = np.array(values[key])
    return Quantization(**values)


def compile_model(model, **kwargs):
    """Compile an in-memory ONNX model through the scheduler (temp file, no repo writes)."""
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder) / (model.graph.name + ".onnx")
        onnx.save(model, path)
        binary, meta = compile_sequence(path, **kwargs)
    return binary, meta


def profile_of(meta):
    """The profile name the scheduler picked, whatever key the emitter used."""
    for key in PROFILE_KEYS:
        if meta.get(key):
            return str(meta[key])
    if meta.get("pool_stages"):
        return "conv-pool-terminal"
    if "limitations" in meta:
        return "legacy-conv"
    return "unknown"


def deterministic_cases(seed, count, height, width, channels):
    """count packed HWC UINT8 cases with fixed corners (0, 255, 128)."""
    rng = np.random.default_rng(seed)
    cases = rng.integers(0, 256, (count, height, width, channels), dtype=np.uint8)
    cases[0], cases[1], cases[2] = 0, 255, 128
    return cases


def byte_round_trip(grid):
    """Write an INT8 grid as .i8 bytes and read them back (what the board compares)."""
    grid = np.ascontiguousarray(grid, dtype=np.int8)
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder) / "expected.i8"
        grid.tofile(path)
        published = np.fromfile(path, dtype=np.int8).reshape(grid.shape)
    return grid, published


print("open-rknpu walkthrough setup complete")

## 1. Compile a single Conv

`open_rknpu.scheduler.compile_sequence` tries every profile in a fixed order and
returns the first that accepts the graph, plus metadata naming it. A 16x16 RGB
image with a 3x3 kernel is outside the legacy 8x8 image profile, so the scheduler
picks the `native16-input` profile.

The compile writes its ONNX model to a temporary directory; nothing is written into
the repository.

In [ ]:
rng = np.random.default_rng(20240501)
single_model = model_graph(
    [conv_node("input", "output", "w", "b", 3)],
    "single_conv",
    [tensor_info("input", [1, 3, 16, 16])],
    [tensor_info("output", [1, 8, 16, 16])],
    [initializer("w", rng.uniform(-0.25, 0.25, (8, 3, 3, 3)).astype(np.float32)),
     initializer("b", rng.uniform(-0.5, 0.5, 8).astype(np.float32))])

single_binary, single_meta = compile_model(single_model, input_scale=1 / 255, input_zero_point=0)
single_info = decode_sequence(single_binary)
print("profile        :", profile_of(single_meta))
print("container bytes:", len(single_binary), "| tasks:", single_info["task_count"])
print("input -> output:", single_info["shape_nhwc"], "->", single_info["output_shape_nhwc"])
print("input layout   :", single_info["input_layout"])
print("input band     : %.6f @ %d" % (single_info["input_scale"], single_info["input_zero_point"]))
print("output band    : %.6f @ %d" % (single_info["output_scale"], single_info["output_zero_point"]))

## 2. Inspect the container

A compiled container is a versioned header, a task table and a payload. The
compiler's own decoder (`open_rknpu.sequence.decode_sequence`) is the parser the
runtime and `tests/board_io.c` agree with, so decoding it here is the same view the
board gets.

In [ ]:
summary = {key: single_info[key] for key in
           ("target", "format_version", "task_count", "input_bytes", "output_bytes",
            "input_stride", "payload_bytes", "arena_bytes", "input_offset", "output_offset",
            "task_bytes")}
print(json.dumps(summary, indent=2))
task = single_info["tasks"][0]
print("first task: command_offset=%d registers=%d enable=%d mask=%d" %
      (task["command_offset"], task["register_count"], task["enable"], task["mask"]))

## 3. A chain with an interior pool

A `Conv -> Relu -> MaxPool -> Conv` graph has a pool that is *not* the last node, so
no fixed-shape profile matches it. The op-level walk accepts it (profile
`chain-walk`): each Conv becomes a native16 task and the pool runs in the same
program, halving 8x8 to 4x4 while preserving the activation band.

In [ ]:
rng = np.random.default_rng(20240502)
walk_model = model_graph(
    [conv_node("input", "c0", "w0", "b0", 3),
     h.make_node("Relu", ["c0"], ["r0"]),
     pool_node("MaxPool", "r0", "pool"),
     conv_node("pool", "output", "w1", "b1", 3)],
    "interior_pool_chain",
    [tensor_info("input", [1, 3, 8, 8])],
    [tensor_info("output", [1, 3, 4, 4])],
    [initializer("w0", rng.uniform(-0.3, 0.3, (8, 3, 3, 3)).astype(np.float32)),
     initializer("b0", rng.uniform(-1.0, 1.0, 8).astype(np.float32)),
     initializer("w1", rng.uniform(-0.3, 0.3, (3, 8, 3, 3)).astype(np.float32)),
     initializer("b1", rng.uniform(-1.0, 1.0, 3).astype(np.float32))])

walk_binary, walk_meta = compile_model(walk_model)
walk_plan = parse_chain(walk_model.graph)
walk_ops = walk_plan["ops"]
walk_info = decode_sequence(walk_binary)
print("profile        :", profile_of(walk_meta))
print("ops            :", [op["kind"] for op in walk_ops])
print("container bytes:", len(walk_binary), "| tasks:", walk_info["task_count"])
print("input -> output:", walk_info["shape_nhwc"], "->", walk_info["output_shape_nhwc"])

## 4. A diamond (fan-out / fan-in)

A 1x1 Conv stem fans out to two heads that are joined by `Add`. The walk handles
the diamond op by op (profile `walk-join`); a three-or-more-head join chain is
emitted by the left-fold chain profile instead. The join forces both heads onto one
scale before they are added.

In [ ]:
rng = np.random.default_rng(20240503)
diamond_nodes = [conv_node("input", "stem0", "w1", "b1", 1),
                 h.make_node("Relu", ["stem0"], ["stem"])]
diamond_constants = [initializer("w1", rng.uniform(-0.3, 0.3, (8, 3, 1, 1)).astype(np.float32)),
                     initializer("b1", rng.uniform(-1.0, 1.0, 8).astype(np.float32))]
for head, kernel in enumerate((3, 1)):
    diamond_constants += [initializer("wh%d" % head,
                                      rng.uniform(-0.3, 0.3, (3, 8, kernel, kernel)).astype(np.float32)),
                          initializer("bh%d" % head, rng.uniform(-1.0, 1.0, 3).astype(np.float32))]
    diamond_nodes.append(conv_node("stem", "h%d" % head, "wh%d" % head, "bh%d" % head, kernel))
diamond_nodes.append(h.make_node("Add", ["h0", "h1"], ["output"]))
diamond_model = model_graph(diamond_nodes, "diamond",
                            [tensor_info("input", [1, 3, 8, 8])],
                            [tensor_info("output", [1, 3, 8, 8])], diamond_constants)

diamond_binary, diamond_meta = compile_model(diamond_model)
diamond_info = decode_sequence(diamond_binary)
print("profile        :", profile_of(diamond_meta))
print("container bytes:", len(diamond_binary), "| tasks:", diamond_info["task_count"])
print("output band    : %.6f @ %d" % (diamond_info["output_scale"], diamond_info["output_zero_point"]))

## 5. Run the profile's Python integer reference and compare bytes

Compilation alone proves nothing. Each profile has an integer reference that
replays the hardware arithmetic - accumulator, multiplier/shift requantization,
clipping - for the *same* quantization parameters the container carries. The
correctness criterion used throughout this repository is **byte equality** between
that reference and the `expected*.i8` bytes a board run compares against.

Here we run `native_input_reference`, `chain_walk_reference` and
`join_walk_reference` on four deterministic cases, write their INT8 bytes, read
them back, and require the bytes to be identical. When the repository's recorded
board suite is next to the notebook, the cell also replays
`research/walk_chain_suite/model000` and asserts the reference reproduces the
recorded board bytes exactly.

In [ ]:
cases = deterministic_cases(20240504, 4, 8, 8, 3)

single_reference = np.stack([
    native_input_reference(case, quantization_of(single_meta["quantization"]),
                           int(single_meta["input_zero_point"]), pads=single_meta["conv_pads"],
                           strides=tuple(single_meta["conv_strides"]),
                           dilations=tuple(single_meta["conv_dilations"]))
    for case in cases])

walk_reference = np.stack([
    chain_walk_reference(case, load_quantizations(walk_meta), walk_ops) for case in cases])

join_plan = parse_join_walk(diamond_model.graph)
join_quantizations = {name: (quantization_of(params) if params else None)
                      for name, params in diamond_meta["quantizations"].items()}
join_reference = np.stack([
    join_walk_reference(case, join_quantizations, join_plan, diamond_meta["join_zero_point"])
    for case in cases])

print("%-24s %7s  %-16s %s" % ("profile reference", "bytes", "sha256[:16]", "bytes equal"))
for label, grid in (("native_input_reference", single_reference),
                    ("chain_walk_reference", walk_reference),
                    ("join_walk_reference", join_reference)):
    grid, published = byte_round_trip(grid)
    digest = hashlib.sha256(grid.tobytes()).hexdigest()[:16]
    equal = bool((grid == published).all())
    print("%-24s %7d  %-16s %s" % (label, grid.size, digest, equal))
    assert equal, "%s: reference bytes did not round-trip" % label

recorded = Path("research/walk_chain_suite")
if recorded.is_dir():
    recorded_model = onnx.load(recorded / "model000.onnx")
    recorded_cases = np.fromfile(recorded / "input000.u8", dtype=np.uint8).reshape(-1, 8, 8, 3)
    recorded_expected = np.fromfile(recorded / "expected000.i8", dtype=np.int8)
    _, recorded_meta = compile_model(recorded_model)
    recorded_ops = parse_chain(recorded_model.graph)["ops"]
    replayed = np.stack([chain_walk_reference(case, load_quantizations(recorded_meta), recorded_ops)
                         for case in recorded_cases]).reshape(-1)
    assert np.array_equal(replayed, recorded_expected), "reference disagrees with board evidence"
    print("board evidence: research/walk_chain_suite/model000 exact (%d bytes)" % replayed.size)
else:
    print("board evidence: research/walk_chain_suite is not next to this notebook")

## 6. Calibrate a small graph with all three methods

Trained networks need measured activation bands; a fixed analytic band loses
accuracy. `open_rknpu.calibration.measure` runs the ONNX host reference over a
directory of `.npy` sample batches and selects a per-Conv-output range with one of
three methods:

* `minmax` - the observed minimum and maximum (one pass, exact for the samples);
* `percentile` - the observed range with the upper tail clipped (here 99.9);
* `kl` - a TensorRT-style saturation search that minimises the KL divergence
  between the full histogram and a 128-level quantization of a candidate range.

Calibration changes the *quality*, never the reference semantics: every variant is
still asserted byte-exact against the same integer pipeline (`quantization.reference`
for the first layer, then `chain.native_reference`).

In [ ]:
rng = np.random.default_rng(20240505)
cal_model = model_graph(
    [conv_node("input", "c1", "w1", "b1", 1),
     h.make_node("Relu", ["c1"], ["r1"]),
     conv_node("r1", "output", "w2", "b2", 1)],
    "calibration",
    [tensor_info("input", [1, 3, 8, 8])],
    [tensor_info("output", [1, 3, 8, 8])],
    [initializer("w1", rng.uniform(-0.7, 0.8, (5, 3, 1, 1)).astype(np.float32)),
     initializer("b1", rng.uniform(-2.0, 2.0, 5).astype(np.float32)),
     initializer("w2", rng.uniform(-0.7, 0.8, (3, 5, 1, 1)).astype(np.float32)),
     initializer("b2", rng.uniform(-2.0, 2.0, 3).astype(np.float32))])

samples = np.random.default_rng(20240506).integers(0, 256, (32, 3, 8, 8), dtype=np.uint8)
with tempfile.TemporaryDirectory() as folder:
    folder = Path(folder)
    model_path = folder / "model.onnx"
    onnx.save(cal_model, model_path)
    calibration_dir = folder / "calibration"
    calibration_dir.mkdir()
    np.save(calibration_dir / "calibration.npy", samples)
    reports = {
        "minmax": measure(model_path, calibration_dir, method="minmax"),
        "percentile": measure(model_path, calibration_dir, method="percentile", percentile=99.9),
        "kl": measure(model_path, calibration_dir, method="kl", bins=512),
    }
print("measured %d calibration samples per method: %s" %
      (reports["minmax"]["samples"], ", ".join(sorted(reports))))

In [ ]:
band_cases = deterministic_cases(20240507, 4, 8, 8, 3)


def calibrated_reference(case, meta):
    """quantization.reference for the first layer, then chain.native_reference."""
    first = quantization_of(meta["first"]["quantization"])
    grid = reference(case, first)
    return native_reference(grid, quantization_of(meta["second"]), first.output_zero_point)


print("%-10s %-20s %8s %10s  %s" % ("variant", "output band (scale@zp)", "container", "int8 bytes",
                                    "reference sha256[:12]"))
variants = [("analytic", None)] + [(name, report["ranges"]) for name, report in reports.items()]
for label, ranges in variants:
    kwargs = {} if ranges is None else {"calibration_ranges": ranges}
    binary, meta = compile_model(cal_model, **kwargs)
    grid = np.stack([calibrated_reference(case, meta) for case in band_cases]).astype(np.int8)
    grid, published = byte_round_trip(grid)
    digest = hashlib.sha256(grid.tobytes()).hexdigest()[:12]
    assert (grid == published).all(), label
    print("%-10s %8.4f @ %4d %8d B %10d  %s" %
          (label, meta["output_scale"], meta["output_zero_point"], len(binary), grid.size, digest))
print("every variant is byte-exact against its own integer reference")

## 7. Where to go next

* `docs/getting-started.md` - the install, the CLI and the options table.
* `docs/architecture.md` - scheduler dispatch order and the full profile list.
* `docs/primitives.md` - every primitive and its verified bounds.
* `docs/verification.md` - the suite hashes, the campaign sweep and the test suite.
* `examples/primitives/README.md` - eleven runnable per-op examples; each one
  publishes a board suite exactly like cells 1-6 here.
* `examples/notebooks/README.md` - how to open this notebook.

## 8. Run it on the board (needs hardware - not executed here)

**Everything below needs the Luckfox Pico Mini B (RV1103). It is not runnable in
this notebook.** These are the exact commands from `docs/board.md`; the board is
also the camera appliance, so `rkipc` must stay alive, and a malformed register
program can wedge the NPU (the recovery is a reboot).

Read the board and stage a suite:

```sh
adb devices                       # the board shows as a USB device, e.g. 498063e3262e55b7
adb shell 'pidof rkipc; free -m; df -h /userdata'
```

Build the libc-only runtime with the ARM uClibc cross compiler from
`research/toolchain/` (`research/fetch_toolchain.py` fetches it; it is not in the
repository):

```sh
research/toolchain/bin/arm-rockchip830-linux-uclibcgnueabihf-gcc \
  --sysroot="$PWD/research/toolchain/arm-rockchip830-linux-uclibcgnueabihf/sysroot" \
  -O2 -std=gnu99 -Wall -Wextra -Werror -Iruntime tests/board_io.c runtime/open_rknpu.c \
  -o /tmp/board_io
```

Stage a published suite under `/userdata/open-npu-research/<name>/` and run every
model, comparing each external output byte with `expectedNNN.i8`:

```sh
PYTHONPATH=src python research/run_v5_suite.py mel_kws_suite --binary /tmp/board_io
```

`make board-suite SUITE=<name>` does the cross-compile and the run in one step
(see the `Makefile`); `docs/board.md` records the timing method, the driver
capabilities that are absent, and the safety notes.